# 07. Azure Synapse Analytics & Storage Setup

## 1. What is Azure Synapse Analytics?
* Formerly called **Azure SQL Data Warehouse**.
* It is an **end-to-end enterprise analytics solution**, not just a standalone data warehouse.
* Unlike other clouds where you need separate services for storage, orchestration, and processing (e.g., AWS Redshift only stores the Core layer), Synapse combines everything in one workspace:
  * **Orchestration / ETL:** Embedded Azure Data Factory (Pipelines).
  * **Data Processing:** Apache Spark Pools & low-code **Mapping Data Flows**.
  * **Data Warehousing:** Serverless & Dedicated SQL Pools (Staging, Core Layer, and Data Marts).

## 2. Azure Free Account & Portal Basics
* **Free Account:** Search *"Azure free account"* → **Try Azure for free**.
  * Provides **$200 USD credit** for **30 days** (cannot be charged after 30 days; services simply nullify).
  * Includes **12 months** of free access to popular services + **always-free** services.
  * Requires a Microsoft account (Gmail/Outlook registered under Microsoft) and card details for verification only.
* **Portal URL:** `portal.azure.com`
* **Resource Group:** A logical **folder** in Azure that holds all your related services/resources (Data Lake, Synapse Analytics, Data Factory, etc.) in one place.

## 3. Step-by-Step Resource Setup

### Step 1: Create a Resource Group
1. Search **Resource groups** in the Azure Portal → Click **+ Create**.
2. **Resource Group Name:** `RG_Azure_Data_Warehouse` *(or any descriptive name)*.
3. **Region:** `Canada East` *(or any available region)*.
4. Click **Review + create** → **Create** → **Go to resource group**.

### Step 2: Create Azure Synapse Analytics Workspace
1. Inside your Resource Group, click **+ Create** → Search **Azure Synapse** → Select **Azure Synapse Analytics** *(by Microsoft)* → **Create**.
2. **Basics Tab:**
   * **Resource Group:** Select the one you just created.
   * **Managed Resource Group:** Leave blank *(Azure auto-generates it)*.
   * **Workspace Name:** Must be globally unique and **lowercase only** *(e.g., `azuredatawarehousing`)*.
   * **Region:** Pick a popular region to avoid networking/availability issues *(e.g., West Europe, UK South, Canada Central, East Asia)*.
   * **Data Lake Storage Gen2 (Account Name):** Synapse requires a primary/managed Data Lake during setup (create new or pick existing).
     * *Best Practice:* In production projects, do **not** use this managed Data Lake; create and connect an **external Data Lake** for better security and governance.
3. **Security Tab:**
   * Create a **SQL Admin username** *(e.g., `admin`)* and a strong **Password**.
4. **Networking Tab:**
   * Enable **Allow all connections** *(for practice)*.
5. **Review + Create:**
   * *Pricing Note:* Serverless SQL Pool costs **~$5 USD per TB** of data processed.
   * Click **Create** *(deployment takes a few minutes)*.

### Step 3: Create an External Data Lake (ADLS Gen2)
1. Go back to your Resource Group → Click **+ Create** → Search **Storage account** *(by Microsoft)* → **Create**.
2. **Basics Tab:**
   * **Storage Account Name:** Must be globally unique *(e.g., `azuredatawarehouselake`)*.
   * **Region:** Same region as your Synapse Workspace.
   * **Performance:** `Standard`.
   * **Redundancy:** **LRS (Locally Redundant Storage)** — the cheapest option.
3. **Advanced Tab (Crucial!):**
   * Check **Enable hierarchical namespace**.
   * *Why?* Without this, it stays a standard Blob Storage account and will not support Data Lake folder structures.
4. Click **Review + create** → **Create** → **Go to resource**.

### Step 4: Configure Containers & Upload Source Data
Inside a Storage Account, **Data storage** provides 4 services:
* **Containers:** Data Lake storage *(used for analytics pipelines)*.
* **File shares:** Central repository to share files with a team *(similar to SharePoint)*.
* **Queues:** Message storage *(similar to Kafka)*.
* **Tables:** Semi-structured NoSQL storage *(similar to Cosmos DB)*.

**Create 4 Containers** (`Data storage` → `Containers` → `+ Container`):
1. `source` — Acting as the raw data source *(using a Data Lake as a source instead of a SQL DB is common in modern pipelines)*.
2. `bronze` — Staging / Raw layer *(as-is data; can be transient/temporary or persistent/permanent)*.
3. `silver` — Transform / Enriched layer.
4. `gold` — Core / Curated layer.

**Upload Initial Data:**
* Open the `source` container → Click **Upload** → Upload the initial batch file (`day0.csv`) first *(keep `day1.csv` aside for testing Incremental Loading later)*.

## 4. Azure Synapse Studio Tour
Open your deployed Synapse Workspace and launch **Synapse Studio**:

| Tab | Purpose & Key Details |
| :--- | :--- |
| **Home** | Quick links for *Ingest*, *Explore & Analyze*, and *Visualize* (rarely used in practice). |
| **Data** | **Workspace:** Create and manage SQL Warehouses & Lake databases.<br>**Linked:** View and connect external Storage Accounts and datasets. |
| **Develop** | Write and save **SQL Scripts** (to build the Data Warehouse), Spark Notebooks, and **Mapping Data Flows**. |
| **Integrate** | **Embedded Azure Data Factory** (just renamed). Build ETL pipelines, Copy activities, and orchestration workflows. |
| **Monitor** | Monitor pipeline runs, SQL requests, and **Apache Spark Pools** (Spark pools are a key feature available here that ADF lacks). |
| **Manage** | Configure **Linked Services**, SQL Pools, Spark Pools, and security. |

### Key Terms to Remember
* **Linked Service:** A connection configuration between Synapse and an external source or destination (e.g., connecting Synapse to an ADLS Gen2 Data Lake).
* **Mapping Data Flows:** A **low-code / no-code** drag-and-drop transformation tool in Synapse. Behind the scenes, it automatically runs **Apache Spark** code.